# Myst Architecture KEM Simulation (liboqs + Shamir's Secret Sharing)

This notebook simulates the "Touch of Evil" backdoor-tolerant architecture using the **liboqs-python** library for ML-KEM (Kyber512). 

We use Shamir's Secret Sharing (SSS) over a 257-bit prime to chunk, distribute, and reconstruct the 1632-byte Kyber512 secret key across a quorum of untrusted ICs.

In [1]:
import os
import time
import random
import oqs

# --- SHAMIR'S SECRET SHARING MATH ---
# We use a 257-bit prime to safely hold 32-byte (256-bit) chunks of the Kyber key.
PRIME = 2**256 + 297 

def eval_poly(poly, x):
    """Evaluates a polynomial at x using Horner's method, modulo PRIME."""
    result = 0
    for coeff in reversed(poly):
        result = (result * x + coeff) % PRIME
    return result

def split_secret_chunk(secret_int, t, n):
    """Splits a single integer into n shares, requiring t to reconstruct."""
    poly = [secret_int] + [random.randint(1, PRIME - 1) for _ in range(t - 1)]
    shares = []
    for i in range(1, n + 1):
        shares.append((i, eval_poly(poly, i)))
    return shares

def lagrange_interpolate(shares):
    """Reconstructs the secret integer at x=0 using Lagrange interpolation."""
    secret = 0
    for i, (x_i, y_i) in enumerate(shares):
        numerator, denominator = 1, 1
        for j, (x_j, y_j) in enumerate(shares):
            if i != j:
                numerator = (numerator * -x_j) % PRIME
                denominator = (denominator * (x_i - x_j)) % PRIME
        
        inv_denominator = pow(denominator, PRIME - 2, PRIME)
        lagrange = (y_i * numerator * inv_denominator) % PRIME
        secret = (secret + lagrange) % PRIME
        
    return (secret + PRIME) % PRIME

print("Libraries imported and Shamir's math initialized.")

Libraries imported and Shamir's math initialized.


In [2]:
# --- MYST ARCHITECTURE COMPONENTS ---

class ProcessingIC:
    """An untrusted IC that holds a Shamir share of the chunks of the Secret Key."""
    def __init__(self, ic_id):
        self.ic_id = ic_id
        self.sk_shares = [] 

    def store_shares(self, shares):
        self.sk_shares = shares

    def request_shares(self):
        return self.sk_shares


class MystThresholdSystem:
    def __init__(self, total_ics=5, threshold=3):
        self.total_ics = total_ics
        self.threshold = threshold
        self.quorum = [ProcessingIC(i) for i in range(total_ics)]
        self.public_key = None
        self.kemalg = "Kyber512"

    def log_phase(self, phase_name, start_time):
        elapsed = (time.perf_counter() - start_time) * 1000
        print(f"[LOG] {phase_name} completed in {elapsed:.3f} ms")

print("Architecture classes defined.")

Architecture classes defined.


### Phase 1: Key Generation & Distributed Sharing
The host generates the liboqs Kyber keypair, extracts the secret bytes, chunks them, and distributes the shares before clearing the original object.

In [3]:
myst_system = MystThresholdSystem(total_ics=5, threshold=3)

print(f"\n--- PHASE 1: Keygen & Shamir Distribution ({myst_system.threshold}-of-{myst_system.total_ics}) ---")
start_time = time.perf_counter()

# 1. Generate keys using liboqs
kem_master = oqs.KeyEncapsulation(myst_system.kemalg)
myst_system.public_key = kem_master.generate_keypair()
secret_key_bytes = kem_master.secret_key

# 2. Chunk the secret key into 32-byte segments
chunk_size = 32
sk_chunks = [secret_key_bytes[i:i+chunk_size] for i in range(0, len(secret_key_bytes), chunk_size)]

# 3. Split each chunk using SSS and distribute to ICs
ic_share_lists = [[] for _ in range(myst_system.total_ics)] 

for chunk in sk_chunks:
    chunk_int = int.from_bytes(chunk, byteorder='big')
    shares = split_secret_chunk(chunk_int, myst_system.threshold, myst_system.total_ics)
    
    for i in range(myst_system.total_ics):
        ic_share_lists[i].append(shares[i])
        
# 4. Load shares into ICs
for i in range(myst_system.total_ics):
    myst_system.quorum[i].store_shares(ic_share_lists[i])
    
# Securely delete the master secret key from the host
kem_master.free() 
del secret_key_bytes
del sk_chunks

myst_system.log_phase("Distributed Key Generation & Splitting", start_time)


--- PHASE 1: Keygen & Shamir Distribution (3-of-5) ---
[LOG] Distributed Key Generation & Splitting completed in 13.879 ms


### Phase 2: Host Encapsulation
We use a fresh `liboqs` object to simulate the sender encapsulating a secret using the public key.

In [4]:
print("\n--- PHASE 2: Host Encapsulation ---")
start_time = time.perf_counter()

kem_sender = oqs.KeyEncapsulation(myst_system.kemalg)
ciphertext, sent_secret = kem_sender.encap_secret(myst_system.public_key)
kem_sender.free()

myst_system.log_phase("Encapsulation", start_time)
print(f"[*] Sent Shared Secret (first 16 bytes): {sent_secret[:16].hex()}...")


--- PHASE 2: Host Encapsulation ---
[LOG] Encapsulation completed in 0.548 ms
[*] Sent Shared Secret (first 16 bytes): e6eaf5c64f994618613dc2a470873c4f...


### Phase 3: Threshold Reconstruction & Decapsulation
The controller rebuilds the key bytes from the threshold ICs, injects it into a new `liboqs` object, and performs standard decapsulation.

In [7]:
import ctypes

print("\n--- PHASE 3: Threshold Reconstruction & Decapsulation ---")
start_time = time.perf_counter()

# Choose exactly threshold number of ICs (e.g., ICs 0, 2, and 4)
participating_ic_indices = [0, 2, 4]
print(f"[*] Reconstructing Kyber key using ICs: {participating_ic_indices}")

# 1. Gather shares
gathered_shares = [myst_system.quorum[i].request_shares() for i in participating_ic_indices]

# 2. Reconstruct the Secret Key chunk by chunk
num_chunks = len(gathered_shares[0])
reconstructed_sk_bytes = bytearray()

for chunk_idx in range(num_chunks):
    chunk_shares = [gathered_shares[ic_idx][chunk_idx] for ic_idx in range(len(participating_ic_indices))]
    reconstructed_int = lagrange_interpolate(chunk_shares)
    reconstructed_sk_bytes.extend(reconstructed_int.to_bytes(32, byteorder='big'))
    
reconstructed_sk = bytes(reconstructed_sk_bytes)

# 3. Run standard liboqs decapsulation with ctypes casting
try:
    kem_receiver = oqs.KeyEncapsulation(myst_system.kemalg)
    
    # FIX: Cast the raw Python bytes into the specific ctypes buffer liboqs expects
    sk_len = kem_receiver.details['length_secret_key']
    c_type_buffer = (ctypes.c_ubyte * sk_len).from_buffer_copy(reconstructed_sk)
    
    # Inject the correctly formatted C-buffer
    kem_receiver.secret_key = c_type_buffer  
    
    rcvd_secret = kem_receiver.decap_secret(ciphertext)
    kem_receiver.free()
    
except Exception as e:
    print(f"[CRITICAL] Standard Kyber decapsulation failed. Keys likely corrupted: {e}")
    rcvd_secret = None

myst_system.log_phase("Reconstruction & Decapsulation", start_time)


--- PHASE 3: Threshold Reconstruction & Decapsulation ---
[*] Reconstructing Kyber key using ICs: [0, 2, 4]
[LOG] Reconstruction & Decapsulation completed in 26.551 ms


### Phase 4: Verification
Strict comparison to ensure the original secret is intact.

In [8]:
print("\n--- PHASE 4: Verification ---")
start_time = time.perf_counter()

if rcvd_secret is None:
    print("[FAIL] No secret returned.")
elif sent_secret == rcvd_secret:
    print(f"[SUCCESS] The decapsulated secret matches the sent secret perfectly!")
    print(f"[*] Sent Shared Secret: {sent_secret.hex()}")
    print(f"[*] Rcvd Shared Secret: {rcvd_secret.hex()}")
else:
    print("[FAIL] Mismatch! The reconstructed key yielded a different secret.")
    
myst_system.log_phase("Verification", start_time)


--- PHASE 4: Verification ---
[SUCCESS] The decapsulated secret matches the sent secret perfectly!
[*] Sent Shared Secret: e6eaf5c64f994618613dc2a470873c4f82af8ebcbce1a27b66a606e37ce7eec4
[*] Rcvd Shared Secret: e6eaf5c64f994618613dc2a470873c4f82af8ebcbce1a27b66a606e37ce7eec4
[LOG] Verification completed in 0.214 ms
